# DSCI 552 — Homework 3

In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download the AReM data

The AReM dataset was downloaded from the UCI Machine Learning Repository and unzipped into `./AReM`.
It contains 7 folders (`bending1`, `bending2`, `cycling`, `lying`, `sitting`, `standing`, `walking`);
each CSV file is one instance with 6 time series
(`avg_rss12`, `var_rss12`, `avg_rss13`, `var_rss13`, `avg_rss23`, `var_rss23`) of 480 values.

**Data cleaning.** A few files are not in the standard format, so the loader below handles them:

| File | Issue | Fix |
|---|---|---|
| `bending2/dataset4.csv` | Space-delimited instead of comma-delimited (with trailing spaces) | Split on commas *or* whitespace |
| `cycling/dataset9.csv`, `cycling/dataset14.csv` | Trailing comma on the last row (creates an empty 8th column) | Drop empty tokens, keep the first 7 columns |
| `sitting/dataset8.csv` | 479 rows instead of 480 (timestamp 13500 is missing) | Kept as observed: statistics use 479 samples; the missing value is unknown, so its effect cannot be determined exactly |

In [2]:
DATA_DIR = "AReM"   # path to the unzipped AReM folder

ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
COLUMNS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES = COLUMNS[1:]


def load_instance(path):
    """Read one AReM file: skip '#' header lines, accept comma- or space-delimited rows."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tokens = [t for t in re.split(r"[,\s]+", line) if t]
            rows.append([float(t) for t in tokens[:7]])
    return pd.DataFrame(rows, columns=COLUMNS)


def file_number(name):
    return int(re.findall(r"\d+", name)[0])


instances = []   # list of dicts: activity, dataset number, split, data
for activity in ACTIVITIES:
    files = sorted((f for f in os.listdir(os.path.join(DATA_DIR, activity)) if f.endswith(".csv")),
                   key=file_number)
    for f in files:
        instances.append({
            "activity": activity,
            "dataset": file_number(f),
            "data": load_instance(os.path.join(DATA_DIR, activity, f)),
        })

print("Number of instances:", len(instances))
print("Instances per activity:", pd.Series([d["activity"] for d in instances]).value_counts().sort_index().to_dict())
print("Instances without 480 rows:",
      [(d["activity"], d["dataset"], len(d["data"])) for d in instances if len(d["data"]) != 480])
print("Missing values after parsing:", sum(int(d["data"].isna().sum().sum()) for d in instances))

Number of instances: 88
Instances per activity: {'bending1': 7, 'bending2': 6, 'cycling': 15, 'lying': 15, 'sitting': 15, 'standing': 15, 'walking': 15}
Instances without 480 rows: [('sitting', 8, 479)]
Missing values after parsing: 0


### (b) Train/test split

Datasets 1 and 2 in `bending1` and `bending2`, and datasets 1, 2, and 3 in every other folder, are the test set.
All remaining datasets are the training set.

In [3]:
def is_test(activity, dataset):
    if activity in ("bending1", "bending2"):
        return dataset in (1, 2)
    return dataset in (1, 2, 3)


for d in instances:
    d["split"] = "test" if is_test(d["activity"], d["dataset"]) else "train"

split_table = (pd.DataFrame([(d["activity"], d["split"]) for d in instances], columns=["activity", "split"])
                 .value_counts().unstack(fill_value=0))
split_table.loc["Total"] = split_table.sum()
split_table

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


#### i. Time-domain features commonly used in time series classification

- **Location:** mean and median.
- **Extremes and spread:** minimum, maximum, range, variance, standard deviation, and interquartile range.
- **Order statistics:** first quartile, third quartile, and other percentiles.
- **Distribution shape:** skewness and kurtosis.
- **Magnitude:** root mean square (RMS) and signal energy (sum of squares).
- **Temporal dynamics:** autocorrelation, zero- or mean-crossing rate, number of peaks, linear trend, and mean of first differences.
- **Complexity:** sample entropy and approximate entropy.

These describe signals directly in the time domain, without requiring a Fourier transform.

**Reference:** tsfresh documentation, [Overview on extracted features](https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html) (accessed October 12, 2026).


#### ii. Extract min, max, mean, median, standard deviation, first quartile, and third quartile

For each of the 88 instances, I compute the 7 features for each of the 6 time series, which gives 42 features.
Feature names follow the homework's convention: e.g., `1st_quart6` is the first quartile of the 6th time series (`var_rss23`).
I keep the features in their original units (not standardized), because part iii asks for the standard deviation of each feature,
which would trivially equal 1 after standardization.

In [4]:
FEATURE_FUNCS = {
    "min": np.min,
    "max": np.max,
    "mean": np.mean,
    "median": np.median,
    "std": lambda x: np.std(x, ddof=1),
    "1st_quart": lambda x: np.percentile(x, 25),
    "3rd_quart": lambda x: np.percentile(x, 75),
}


def extract_features(df):
    feats = {}
    for i, col in enumerate(SERIES, start=1):
        x = df[col].to_numpy()
        for name, func in FEATURE_FUNCS.items():
            feats[f"{name}{i}"] = func(x)
    return feats


features = pd.DataFrame([extract_features(d["data"]) for d in instances])
features.index = pd.RangeIndex(1, len(features) + 1, name="Instance")
meta = pd.DataFrame({"activity": [d["activity"] for d in instances],
                     "dataset": [d["dataset"] for d in instances],
                     "split": [d["split"] for d in instances]}, index=features.index)

print("Feature matrix shape:", features.shape)
features.round(3)

Feature matrix shape: (88, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.625,40.500,1.477,39.250,42.000,0.0,1.30,0.359,0.430,0.323,0.000,0.500,4.00,29.50,19.041,19.250,4.463,16.000,23.250,0.0,7.23,0.833,0.500,0.966,0.000,1.120,27.25,38.25,34.311,35.000,2.188,33.000,36.000,0.00,1.92,0.571,0.430,0.583,0.000,1.300
2,38.00,45.67,42.813,42.500,1.436,42.000,43.670,0.0,1.22,0.372,0.470,0.289,0.000,0.500,2.00,29.50,20.097,21.000,3.894,19.000,22.250,0.0,5.76,0.876,0.500,1.047,0.000,1.145,27.67,38.50,33.025,33.000,1.995,32.000,34.500,0.00,3.11,0.571,0.430,0.601,0.000,1.300
3,35.00,47.40,43.954,44.330,1.559,43.000,45.000,0.0,1.70,0.426,0.470,0.339,0.000,0.500,6.50,29.75,22.122,23.000,3.031,19.750,24.000,0.0,4.44,0.497,0.430,0.551,0.000,0.830,29.00,38.50,35.588,36.000,2.000,35.362,36.500,0.00,1.79,0.493,0.430,0.514,0.000,0.940
4,33.00,47.75,42.180,43.500,3.671,39.150,45.000,0.0,3.00,0.696,0.500,0.631,0.000,1.120,8.50,30.00,22.184,23.000,3.810,20.500,24.372,0.0,5.15,0.990,0.830,0.954,0.430,1.300,20.00,38.67,33.494,35.000,3.849,30.458,36.330,0.00,2.18,0.614,0.500,0.524,0.000,1.000
5,33.00,45.75,41.678,41.750,2.243,41.330,42.750,0.0,2.83,0.536,0.500,0.405,0.430,0.710,3.00,28.25,19.007,19.125,4.087,16.500,22.062,0.0,6.42,0.842,0.500,0.929,0.430,1.120,23.67,37.50,29.857,30.000,2.411,28.458,31.250,0.00,1.79,0.383,0.430,0.389,0.000,0.500
6,37.00,48.00,43.455,43.250,1.386,42.500,45.000,0.0,1.58,0.378,0.470,0.316,0.000,0.500,5.75,27.00,15.793,15.000,3.848,13.000,18.270,0.0,10.03,0.849,0.500,0.996,0.430,1.120,8.00,33.50,23.035,23.500,2.489,22.250,24.000,0.00,5.26,0.680,0.500,0.623,0.430,0.870
7,36.25,48.00,43.969,44.500,1.618,43.310,44.670,0.0,1.50,0.413,0.470,0.263,0.430,0.500,1.50,26.33,15.868,16.250,3.742,14.250,18.000,0.0,5.17,0.666,0.470,0.789,0.000,0.940,11.33,30.75,22.104,21.670,3.318,20.500,23.750,0.00,2.96,0.555,0.490,0.488,0.000,0.830
8,12.75,51.00,24.563,24.250,3.738,23.188,26.500,0.0,6.87,0.591,0.430,0.837,0.000,0.710,0.00,25.33,19.121,20.250,4.300,18.000,22.000,0.0,6.76,0.737,0.470,0.927,0.000,0.870,10.67,30.00,23.493,23.750,3.694,20.500,27.000,0.00,4.97,0.700,0.500,0.694,0.430,0.870
9,0.00,42.75,27.465,28.000,3.584,25.500,30.000,0.0,7.76,0.450,0.430,0.767,0.000,0.500,7.50,35.00,20.843,20.750,3.826,18.000,22.500,0.0,5.76,0.780,0.500,0.904,0.000,0.940,0.00,33.00,17.618,18.000,5.054,15.000,20.750,0.00,6.76,1.122,0.830,1.012,0.470,1.300


In [5]:
# Same table with labels, for later use in classification (Homework 4)
dataset_full = pd.concat([features, meta], axis=1)
dataset_full.head()

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,dataset,split
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,0.43,0.322605,0.00,0.50,4.0,29.50,19.040937,19.250,4.462952,16.00,23.2500,0.0,7.23,0.832542,0.50,0.965659,0.00,1.120,27.25,38.25,34.311292,35.0,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30,bending1,1,test
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,0.47,0.289158,0.00,0.50,2.0,29.50,20.096979,21.000,3.893737,19.00,22.2500,0.0,5.76,0.876438,0.50,1.047259,0.00,1.145,27.67,38.50,33.024583,33.0,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30,bending1,2,test
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,0.47,0.338690,0.00,0.50,6.5,29.75,22.122354,23.000,3.030943,19.75,24.0000,0.0,4.44,0.497312,0.43,0.550657,0.00,0.830,29.00,38.50,35.588458,36.0,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94,bending1,3,train
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,0.50,0.630860,0.00,1.12,8.5,30.00,22.183625,23.000,3.810469,20.50,24.3725,0.0,5.15,0.989917,0.83,0.953730,0.43,1.300,20.00,38.67,33.493917,35.0,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00,bending1,4,train
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,0.50,0.405469,0.43,0.71,3.0,28.25,19.006563,19.125,4.087107,16.50,22.0625,0.0,6.42,0.841875,0.50,0.928801,0.43,1.120,23.67,37.50,29.857083,30.0,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50,bending1,5,train


#### iii. Standard deviation of each feature and 90% bootstrap confidence intervals

For each of the 42 features, I estimate the standard deviation across the 88 instances.
Then I use `scipy.stats.bootstrap` (percentile method, 2,000 resamples) to build a 90% confidence interval for that standard deviation.

In [6]:
rng = np.random.default_rng(552)


def sample_std(x, axis=-1):
    return np.std(x, ddof=1, axis=axis)


rows = []
for col in features.columns:
    x = features[col].to_numpy()
    res = stats.bootstrap((x,), sample_std, confidence_level=0.90, n_resamples=2000,
                          method="percentile", random_state=rng)
    rows.append({"feature": col,
                 "std_estimate": sample_std(x),
                 "CI_90_lower": res.confidence_interval.low,
                 "CI_90_upper": res.confidence_interval.high})

std_ci = pd.DataFrame(rows).set_index("feature")
std_ci.round(4)

,std_estimate,CI_90_lower,CI_90_upper
feature,,,
min1,9.5700,8.2529,10.7615
max1,4.3944,3.3557,5.2685
mean1,5.3357,4.7064,5.8872
median1,5.4401,4.7937,5.9874
std1,1.7722,1.5712,1.9512
1st_quart1,6.1536,5.5678,6.6459
3rd_quart1,5.1389,4.3433,5.8393
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6205,5.4022


**Observations.**
- Many features of the `avg_rss` series (1, 3, 5) have larger across-instance standard deviations than corresponding features of the `var_rss` series (2, 4, 6), but this is not universal. The unstandardized features have different numerical scales, so larger standard deviations alone do not imply greater classification importance.
- `min2` and `min4` have standard deviation 0 and bootstrap CI [0, 0]: they are 0 in every observed instance and cannot distinguish activities in this dataset. `min6` is almost constant as well.
- Here, the standard deviation of `mean1`, for example, describes variation in average signal level across recordings. This differs from `std1`, which measures variation within a recording.


#### iv. Select the three most important time-domain features

I select **mean, standard deviation, and maximum** to represent typical signal level, fluctuation, and peak value. This is a reasoned choice rather than a claim of optimal predictive performance.

To support exploration, I use **only the 69 training instances**, retaining the **seven original activity labels**:
1. **Class separation:** one-way ANOVA F-statistics, averaged over the six time series. Larger F indicates greater between-class variation relative to within-class variation.
2. **Redundancy:** absolute correlations between feature types, averaged over series where both features are nonconstant.

Test data are excluded from this analysis. If feature selection is tuned with cross-validation in Homework 4, it must be repeated within each training fold.



In [7]:
train_mask = meta["split"].eq("train")
features_train = features.loc[train_mask]
activity_train = meta.loc[train_mask, "activity"]
groups = activity_train.unique()


def anova_f(col):
    x = features_train[col]
    if x.std() == 0:
        return 0.0          # constant feature cannot separate classes
    return stats.f_oneway(*[x[activity_train == g] for g in groups]).statistic


f_scores = pd.DataFrame({name: [anova_f(f"{name}{i}") for i in range(1, 7)]
                         for name in FEATURE_FUNCS},
                        index=[f"series {i} ({s})" for i, s in enumerate(SERIES, start=1)]).T
f_scores["average F"] = f_scores.mean(axis=1)
f_scores.sort_values("average F", ascending=False).round(1)

,series 1 (avg_rss12),series 2 (var_rss12),series 3 (avg_rss13),series 4 (var_rss13),series 5 (avg_rss23),series 6 (var_rss23),average F
mean,15.9,900.2,26.4,651.8,33.4,581.5,368.2
3rd_quart,8.5,807.0,28.1,561.9,32.2,664.8,350.4
median,17.5,477.0,19.4,800.8,28.6,563.1,317.7
1st_quart,21.4,265.5,21.2,244.8,25.5,346.3,154.1
std,9.1,360.9,15.2,128.6,7.8,133.4,109.2
max,1.5,47.7,31.7,29.3,37.0,33.0,30.0
min,11.0,0.0,22.7,0.0,24.1,0.8,9.8


In [8]:
names = list(FEATURE_FUNCS)
corr = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        vals = []
        for i in range(1, 7):
            xa, xb = features_train[f"{a}{i}"], features_train[f"{b}{i}"]
            if xa.std() > 0 and xb.std() > 0:
                vals.append(abs(np.corrcoef(xa, xb)[0, 1]))
        corr.loc[a, b] = np.mean(vals)
print("Training-set average |correlation| between feature types (over the time series where both are non-constant)")
corr.round(2)

Training-set average |correlation| between feature types (over the time series where both are non-constant)


,min,max,mean,median,std,1st_quart,3rd_quart
min,1.00,0.36,0.59,0.57,0.40,0.60,0.52
max,0.36,1.00,0.79,0.75,0.63,0.71,0.82
mean,0.59,0.79,1.00,0.99,0.60,0.98,0.98
median,0.57,0.75,0.99,1.00,0.61,0.98,0.96
std,0.40,0.63,0.60,0.61,1.00,0.66,0.57
1st_quart,0.60,0.71,0.98,0.98,0.66,1.00,0.93
3rd_quart,0.52,0.82,0.98,0.96,0.57,0.93,1.00


**Selected features: mean, standard deviation, and maximum.**

- **Mean:** summarizes typical signal level. I choose it as a representative location statistic instead of selecting several location statistics that may be highly correlated.
- **Standard deviation:** measures fluctuations around the mean. Signals with similar average levels can have different variability, making this a useful complement to the mean.
- **Maximum:** captures the highest observed value, providing information about peaks that mean and standard deviation do not fully describe. It is sensitive to outliers, so its predictive value should be assessed during training-set validation.
- **Minimum:** is not selected because it is constant for series 2 and 4 in the observed data, limiting its usefulness across the six series.

ANOVA and correlation are exploratory aids, not proof of optimal classification performance. Correlated features can still add predictive information. The three selected feature types give 18 columns across six time series.


## 2. ISLR 3.7.4

*Setup: n = 100 observations, one predictor. We fit a linear regression $Y=\beta_0+\beta_1X+\varepsilon$ and a cubic regression $Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\varepsilon$.*

**(a) True relationship is linear; training RSS.**
We expect the cubic regression to have a **lower (or equal) training RSS**. The linear model is nested in the cubic model (set $\beta_2=\beta_3=0$),
and least squares minimizes RSS on the training data. So the cubic fit can always do at least as well as the linear fit on the training data.
In practice, the extra terms fit some of the noise $\varepsilon$, so the cubic training RSS will be slightly lower.

**(b) True relationship is linear; test RSS.**
We expect the **linear regression to have a lower test RSS**. Because the true relationship is linear, the linear model has no bias. The cubic model also has no bias, but it has higher variance: its extra coefficients are fit to noise in the training data, and this overfitting does not carry over to new data. So the cubic model's test RSS is expected to be higher.
(For one particular sample this is not guaranteed, but it is what we expect on average.)

**(c) True relationship is not linear; training RSS.**
The **cubic regression will have a lower (or equal) training RSS**. The argument from (a) still holds: the models are nested, so the more flexible cubic model always fits the training data at least as well, whatever the true relationship is.
The size of the gap cannot be determined from nonlinearity alone; it depends on how well a cubic polynomial approximates the true relationship.

**(d) True relationship is not linear; test RSS.**
**There is not enough information to tell.** The answer depends on the bias–variance trade-off:
- If the nonlinearity can be approximated well by a cubic polynomial, the cubic model may have lower test RSS when its reduction in squared bias outweighs its increase in variance.
- If the true relationship is only slightly nonlinear, the cubic model's reduction in bias may be smaller than its increase in variance, and the linear model could have a lower test RSS.

Without knowing the form and strength of the nonlinearity and the noise level, we cannot determine which model will have lower expected test RSS.

*Optional simulation to illustrate the answers (500 repetitions, n = 100 training points, 1,000 test points, noise σ = 1).*

In [9]:
def simulate(f, reps=500, n=100, n_test=1000, sigma=1.0, seed=0):
    r = np.random.default_rng(seed)
    out = np.zeros((reps, 4))
    for k in range(reps):
        x, xt = r.uniform(-2, 2, n), r.uniform(-2, 2, n_test)
        y, yt = f(x) + r.normal(0, sigma, n), f(xt) + r.normal(0, sigma, n_test)
        for j, deg in enumerate((1, 3)):
            coef = np.polyfit(x, y, deg)
            out[k, j] = np.sum((y - np.polyval(coef, x)) ** 2)                 # training RSS
            out[k, 2 + j] = np.sum((yt - np.polyval(coef, xt)) ** 2) * n / n_test  # test RSS (scaled to n=100)
    return pd.Series(out.mean(axis=0), index=["train RSS linear", "train RSS cubic",
                                              "test RSS linear", "test RSS cubic"])


sim = pd.DataFrame({
    "true linear": simulate(lambda x: 1 + 2 * x),
    "slightly nonlinear": simulate(lambda x: 1 + 2 * x + 0.05 * x ** 2),
    "strongly nonlinear": simulate(lambda x: 1 + 2 * x + 1.5 * x ** 2),
}).T
sim.round(2)

,train RSS linear,train RSS cubic,test RSS linear,test RSS cubic
true linear,98.78,96.87,102.25,104.35
slightly nonlinear,99.09,96.87,102.62,104.35
strongly nonlinear,410.00,96.87,430.57,104.35


The simulation illustrates these particular data-generating functions. The cubic model has lower training RSS in all three examples. For a linear truth, the linear model has lower average test RSS. For these quadratic examples, weak curvature favors the linear model and strong curvature favors the cubic model. This does not imply that every strongly nonlinear relationship is well approximated by a cubic polynomial.
